# VisDSR: diagnostics plus an improvement comparison

**One saved run: 192 calls, both models, four matched update methods.**
The original 96 diagnostic calls are retained. An additional 96 compare procedural instructions, constrained JSON, and both together. Original baseline updates are reused by exact provenance. Task difficulty and model revisions are unchanged. The extraction answers also produce a separate model-plus-executor baseline without additional inference.

Enable **Internet** and **GPU T4 ×2**. Use **Save Version → Save & Run All**, wait until Running, then you can sleep your laptop. The two models run independently on separate GPUs and load once each. A single GPU uses sequential execution. A fresh run needs no uploaded ZIP.

Download from the **completed saved version's Output**, which is separate from your interactive editor. Completion and accuracy are measured independently. Hardware or inference failures produce an explicitly partial report and backup. No method is guaranteed to improve accuracy. See the [prospective protocol](https://github.com/Abhishekgit01/VisDSR/blob/main/docs/INTERVENTIONS.md).


In [ ]:
PARALLEL_MODELS = True             # One model per T4; single GPU falls back to sequential
BACKUP_INPUT = "auto"              # Or an explicit diagnostic/improvement ZIP or extracted dataset
if type(PARALLEL_MODELS) is not bool:
    raise ValueError("PARALLEL_MODELS must be True or False.")
print("192 planned responses; baseline requests reused; no repeated calls for cached answers.")


## Setup, validation, and optional restoration

This checks the original frozen inputs and the new intervention freeze. It compiles each task grammar with both actual tokenizers, without downloading model weights. Compatible diagnostic backups from the previous 96-call notebook can be reused. Completed main-study ZIPs belong to a different experiment.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

if not Path("/kaggle/working").is_dir():
    raise RuntimeError("Import this notebook into Kaggle and enable T4 GPUs and Internet.")
if "BACKUP_INPUT" not in globals():
    raise RuntimeError("Run Configuration first.")
PROJECT = Path("/kaggle/working/VisDSR_improvements")
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"
if not PROJECT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/Abhishekgit01/VisDSR.git", str(PROJECT)], check=True)
if not (PROJECT / "interventions/FREEZE.json").is_file():
    raise RuntimeError("This checkout predates the improvement comparison; import the published notebook.")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade-strategy", "only-if-needed",
                "-r", str(PROJECT / "environment/requirements.txt"),
                "-r", str(PROJECT / "environment/requirements-inference.txt"),
                "-r", str(PROJECT / "interventions/requirements.txt")], check=True)
subprocess.run(["make", "build"], cwd=PROJECT, check=True)
for command in (("diagnostics", "check"), ("interventions", "check"), ("interventions", "decoder-check")):
    subprocess.run([sys.executable, "-m", *command], cwd=PROJECT, check=True)

if BACKUP_INPUT == "auto":
    candidates = sorted(Path("/kaggle/input").rglob("*diagnostics*.zip"))
    candidates += sorted(Path("/kaggle/input").rglob("*improvements*.zip"))
    for manifest in sorted(Path("/kaggle/input").rglob("MANIFEST.json")):
        info = json.loads(manifest.read_text())
        if info.get("protocol_id") in ("visdsr-dsu-diagnostics-20261007", "visdsr-dsu-interventions-20261007"):
            candidates.append(manifest.parent)
else:
    candidates = [Path(BACKUP_INPUT)]
for backup in candidates:
    if backup.is_dir():
        info = json.loads((backup / "MANIFEST.json").read_text())
    else:
        import zipfile
        with zipfile.ZipFile(backup) as archive:
            info = json.loads(archive.read("MANIFEST.json"))
    module = "interventions" if info["protocol_id"] == "visdsr-dsu-interventions-20261007" else "diagnostics"
    subprocess.run([sys.executable, "-m", module, "restore", "--input", str(backup)], cwd=PROJECT, check=True)
subprocess.run([sys.executable, "-m", "interventions", "score"], cwd=PROJECT, check=True)
print("Setup complete. Compatible cached answers will be reused.")


## Run diagnostics and all improvement arms

This one cell runs both checkpoints. Every completed answer updates its model's backup; the parent combines both after they exit. It records the diagnostic gate independently of intervention outcomes. Watch for **FINAL STATUS: complete; 192/192 saved**.

A process failure produces **partial** and a resumable combined backup. A finished notebook does not itself establish complete collection or improved accuracy.


In [ ]:
import subprocess
import sys
from pathlib import Path

project = Path("/kaggle/working/VisDSR_improvements")
if "PARALLEL_MODELS" not in globals():
    raise RuntimeError("Run Configuration and Setup first.")
command = [sys.executable, "-u", "-m", "interventions", "overnight", "--output-dir", "/kaggle/working"]
if not PARALLEL_MODELS:
    command.append("--sequential")
subprocess.run(command, cwd=project, check=True)


## Final export

Tomorrow open the **saved version → Output** and read **visdsr_improvements_REPORT.md**.
Download **visdsr_improvements_complete.zip** at 192/192 and send it for review.
For partial collection, download **visdsr_improvements_backup.zip**, attach it as private input, and run this same notebook again. Exact requests are retained and cached answers are skipped.


In [ ]:
import json
import subprocess
import sys
from pathlib import Path

project = Path("/kaggle/working/VisDSR_improvements")
backup = Path("/kaggle/working/visdsr_improvements_backup.zip")
subprocess.run([sys.executable, "-m", "interventions", "export", "--output", str(backup)], cwd=project, check=True)
summary = json.loads((project / "results/followup/visdsr-dsu-interventions-20261007/summary.json").read_text())
print(f"SAVED: {summary['saved']}/192; complete={summary['complete']}")
print(f"BACKUP READY: {backup}; download from the completed saved version's Output.")
